# Phonetics Demo App for Maryland Day
## Interactive Voice Analysis with Streamlit

This notebook contains three interactive demos:
1. **Vowel Plotting** - Record vowels and see F1/F2 formants plotted on a vowel chart
2. **Is This a Question?** - Visualize pitch contours to see rising vs falling intonation
3. **Live Spectrogram & Waveform** - Real-time visualization of your voice

---

## Setup Instructions

### 1. Create Conda Environment (M4 Mac)

```bash
# Create a new conda environment
conda create -n phonetics python=3.10 -y

# Activate the environment
conda activate phonetics

# Install required packages
conda install -c conda-forge numpy scipy matplotlib pandas -y
pip install streamlit praat-parselmouth librosa sounddevice soundfile
```

### 2. Run the Streamlit App

After running all cells in this notebook, a Python file `phonetics_app.py` will be generated.

```bash
streamlit run phonetics_app.py
```

The app will open in your browser at `http://localhost:8501`

---

## Check Requirements

Run this cell to verify all dependencies are installed correctly.

In [1]:
# Check if all required packages are installed
import sys

required_packages = {
    'streamlit': 'streamlit',
    'parselmouth': 'praat-parselmouth',
    'librosa': 'librosa',
    'sounddevice': 'sounddevice',
    'soundfile': 'soundfile',
    'numpy': 'numpy',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'pandas': 'pandas'
}

missing_packages = []

for package, install_name in required_packages.items():
    try:
        __import__(package)
        print(f"✓ {package} is installed")
    except ImportError:
        print(f"✗ {package} is NOT installed")
        missing_packages.append(install_name)

if missing_packages:
    print(f"\n⚠️  Install missing packages with:")
    print(f"pip install {' '.join(missing_packages)}")
else:
    print("\n✅ All required packages are installed!")
    print(f"Python version: {sys.version}")

✓ streamlit is installed
✓ parselmouth is installed
✓ librosa is installed
✓ sounddevice is installed
✓ soundfile is installed
✓ numpy is installed
✓ scipy is installed
✓ matplotlib is installed
✓ pandas is installed

✅ All required packages are installed!
Python version: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 17:06:34) [Clang 19.1.7 ]


---
## Streamlit App Code

The following cell generates the complete Streamlit application with all three demos.

In [ ]:
# Write the Streamlit app to a Python file
app_code = '''
import streamlit as st
import numpy as np
import parselmouth
from import streamlit as st
import numpy as np
import parselmouth
from parselmouth.praat import call
import librosa
import librosa.display
import matplotlib.pyplot as plt
import sounddevice as sd
import soundfile as sf
import os
from datetime import datetime

# Configure page
st.set_page_config(
    page_title="Phonetics Demo - Maryland Day",
    page_icon="🎙️",
    layout="wide"
)

# Color scheme - coordinated colors for visualizations
COLORS = {
    'primary': '#FF6B6B',      # Coral red for waveforms
    'secondary': '#4ECDC4',    # Teal for spectrograms
    'accent': '#FFE66D',       # Yellow for highlights
    'dark': '#2C3E50',         # Dark blue-gray
    'light': '#ECF0F1'         # Light gray
}

# Color palette for multiple vowels
VOWEL_COLORS = ['#FF6B6B', '#4ECDC4', '#95E1D3', '#F38181', '#AA96DA',
                '#FCBAD3', '#A8D8EA', '#FFABAB', '#FFC8A2', '#D4A5A5']

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def record_audio(duration=3, sample_rate=44100):
    """Record audio from the microphone.

    Args:
        duration (int): Recording duration in seconds
        sample_rate (int): Sampling rate in Hz

    Returns:
        tuple: (audio_data, sample_rate)
    """
    with st.spinner(f"🎤 Recording for {duration} seconds..."):
        audio = sd.rec(
            int(duration * sample_rate),
            samplerate=sample_rate,
            channels=1,
            dtype='float64'
        )
        sd.wait()  # Wait until recording is finished
    return audio.flatten(), sample_rate


def save_audio_file(audio, sample_rate, label):
    """Save audio to a WAV file.

    Args:
        audio (np.array): Audio signal
        sample_rate (int): Sampling rate
        label (str): Vowel label for filename

    Returns:
        str: Filename of saved audio
    """
    # Create recordings directory if it doesn't exist
    os.makedirs('recordings', exist_ok=True)

    # Generate filename with timestamp and label
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_label = label.replace(' ', '_').replace('/', '_')
    filename = f"recordings/vowel_{safe_label}_{timestamp}.wav"

    # Save as WAV file
    sf.write(filename, audio, sample_rate)

    return filename


def extract_formants(audio, sample_rate, num_formants=5):
    """Extract formant frequencies using Parselmouth/Praat.

    Args:
        audio (np.array): Audio signal
        sample_rate (int): Sampling rate
        num_formants (int): Number of formants to extract

    Returns:
        dict: Dictionary with F1, F2, F3, etc.
    """
    # Create Praat Sound object
    sound = parselmouth.Sound(audio, sampling_frequency=sample_rate)

    # Extract formants using Praat
    formant = call(sound, "To Formant (burg)", 0.0, num_formants, 5500, 0.025, 50)

    # Get formant values at the middle of the sound
    duration = sound.duration
    mid_point = duration / 2.0

    formants = {}
    for i in range(1, num_formants + 1):
        try:
            f_val = call(formant, "Get value at time", i, mid_point, "Hertz", "Linear")
            if not np.isnan(f_val):
                formants[f"F{i}"] = f_val
        except:
            pass

    return formants


def extract_pitch(audio, sample_rate):
    """Extract pitch contour using Parselmouth/Praat.

    Args:
        audio (np.array): Audio signal
        sample_rate (int): Sampling rate

    Returns:
        tuple: (time_points, pitch_values)
    """
    sound = parselmouth.Sound(audio, sampling_frequency=sample_rate)

    # Extract pitch using autocorrelation (Praat default)
    pitch = call(sound, "To Pitch", 0.0, 75, 600)  # 75-600 Hz range for human speech

    # Get pitch values over time
    time_points = []
    pitch_values = []

    for t in np.arange(0, sound.duration, 0.01):  # Sample every 10ms
        pitch_value = call(pitch, "Get value at time", t, "Hertz", "Linear")
        if not np.isnan(pitch_value) and pitch_value > 0:
            time_points.append(t)
            pitch_values.append(pitch_value)

    return np.array(time_points), np.array(pitch_values)


def plot_vowel_chart_multi(vowel_recordings=None):
    """Plot multiple vowels on a standard F1-F2 vowel chart.

    Args:
        vowel_recordings (list): List of dicts with 'f1', 'f2', 'label' keys

    Returns:
        matplotlib.figure.Figure: The vowel chart figure
    """
    fig, ax = plt.subplots(figsize=(12, 10), facecolor='white')

    # Reference vowels (approximate American English values)
    reference_vowels = {
        'i (heed)': (280, 2250),
        'ɪ (hid)': (400, 1900),
        'e (hayed)': (400, 2100),
        'ɛ (head)': (550, 1800),
        'æ (had)': (700, 1700),
        'ɑ (hod)': (700, 1100),
        'ɔ (hawed)': (600, 900),
        'o (hoed)': (450, 850),
        'ʊ (hood)': (400, 1000),
        'u (who\'d)': (300, 900)
    }

    # Plot reference vowels (lighter, in background)
    for vowel, (f1_ref, f2_ref) in reference_vowels.items():
        ax.scatter(f2_ref, f1_ref, s=200, alpha=0.2, c=COLORS['secondary'],
                  edgecolors=COLORS['dark'], linewidth=1.5)
        ax.annotate(vowel, (f2_ref, f1_ref), fontsize=10, ha='center',
                   va='center', weight='bold', alpha=0.5)

    # Plot user's vowels
    if vowel_recordings:
        for idx, vowel in enumerate(vowel_recordings):
            color = VOWEL_COLORS[idx % len(VOWEL_COLORS)]
            f1 = vowel['f1']
            f2 = vowel['f2']
            label = vowel['label']

            # Plot the vowel point
            ax.scatter(f2, f1, s=400, c=color, marker='*',
                      edgecolors=COLORS['dark'], linewidth=2.5, zorder=10,
                      label=label)

            # Add label with offset to avoid overlap
            offset_y = 25 + (idx % 3) * 15  # Stagger labels
            ax.annotate(label, (f2, f1), xytext=(0, offset_y),
                       textcoords='offset points', fontsize=13,
                       weight='bold', color=color,
                       ha='center',
                       bbox=dict(boxstyle='round,pad=0.5',
                                facecolor='white',
                                edgecolor=color,
                                linewidth=2,
                                alpha=0.9))

    # Invert axes (phonetic convention)
    ax.invert_xaxis()
    ax.invert_yaxis()

    # Labels and formatting
    ax.set_xlabel('F2 (Hz)', fontsize=14, weight='bold')
    ax.set_ylabel('F1 (Hz)', fontsize=14, weight='bold')

    title = 'Vowel Space Chart (F1 vs F2)'
    if vowel_recordings:
        title += f' - {len(vowel_recordings)} vowel(s) recorded'
    ax.set_title(title, fontsize=16, weight='bold', pad=20)

    ax.grid(True, alpha=0.3, linestyle='--')

    if vowel_recordings:
        ax.legend(fontsize=11, loc='upper right', framealpha=0.9)

    plt.tight_layout()
    return fig


def plot_pitch_contour(time, pitch):
    """Plot pitch contour over time.

    Args:
        time (np.array): Time points in seconds
        pitch (np.array): Pitch values in Hz

    Returns:
        matplotlib.figure.Figure: The pitch contour figure
    """
    fig, ax = plt.subplots(figsize=(12, 6), facecolor='white')

    # Plot pitch contour
    ax.plot(time, pitch, linewidth=3, color=COLORS['primary'], label='Pitch')
    ax.fill_between(time, pitch, alpha=0.3, color=COLORS['primary'])

    # Determine if rising or falling
    if len(pitch) > 2:
        # Compare first third vs last third
        third = len(pitch) // 3
        start_avg = np.mean(pitch[:third])
        end_avg = np.mean(pitch[-third:])

        if end_avg > start_avg * 1.1:  # 10% threshold
            pattern = "Rising (Question-like) ⬆️"
            color = COLORS['secondary']
        elif end_avg < start_avg * 0.9:
            pattern = "Falling (Statement-like) ⬇️"
            color = COLORS['accent']
        else:
            pattern = "Level (Flat) ➡️"
            color = COLORS['dark']

        ax.set_title(f'Pitch Contour: {pattern}', fontsize=16, weight='bold',
                    pad=20, color=color)
    else:
        ax.set_title('Pitch Contour', fontsize=16, weight='bold', pad=20)

    # Labels and formatting
    ax.set_xlabel('Time (s)', fontsize=14, weight='bold')
    ax.set_ylabel('Pitch (Hz)', fontsize=14, weight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=12)

    plt.tight_layout()
    return fig


def plot_waveform_and_spectrogram(audio, sample_rate):
    """Plot waveform and spectrogram with coordinated colors.

    Args:
        audio (np.array): Audio signal
        sample_rate (int): Sampling rate

    Returns:
        matplotlib.figure.Figure: Combined waveform and spectrogram figure
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), facecolor='white')

    # Time array
    time = np.linspace(0, len(audio) / sample_rate, len(audio))

    # === Waveform ===
    ax1.plot(time, audio, linewidth=1, color=COLORS['primary'], alpha=0.8)
    ax1.fill_between(time, audio, alpha=0.3, color=COLORS['primary'])
    ax1.set_xlabel('Time (s)', fontsize=12, weight='bold')
    ax1.set_ylabel('Amplitude', fontsize=12, weight='bold')
    ax1.set_title('Waveform', fontsize=14, weight='bold', pad=15)
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_xlim([0, time[-1]])

    # === Spectrogram ===
    # Compute spectrogram using librosa
    D = librosa.stft(audio)
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    # Plot spectrogram with custom colormap
    img = librosa.display.specshow(
        S_db,
        sr=sample_rate,
        x_axis='time',
        y_axis='hz',
        ax=ax2,
        cmap='viridis'  # Beautiful teal-to-yellow colormap
    )

    ax2.set_xlabel('Time (s)', fontsize=12, weight='bold')
    ax2.set_ylabel('Frequency (Hz)', fontsize=12, weight='bold')
    ax2.set_title('Spectrogram', fontsize=14, weight='bold', pad=15)
    ax2.set_ylim([0, 8000])  # Focus on speech range

    # Add colorbar
    cbar = fig.colorbar(img, ax=ax2, format='%+2.0f dB')
    cbar.set_label('Intensity (dB)', fontsize=11, weight='bold')

    plt.tight_layout()
    return fig


# =============================================================================
# STREAMLIT APP
# =============================================================================

def main():
    """Main Streamlit app with three demo modes."""

    # Initialize session state for vowel recordings
    if 'vowel_recordings' not in st.session_state:
        st.session_state.vowel_recordings = []

    # Header
    st.title("🎙️ Phonetics Demo - Maryland Day")
    st.markdown("""
    ### Explore the Science of Speech!
    Choose a demo below to analyze your voice in real-time.
    """)

    # Sidebar for demo selection
    st.sidebar.title("Demo Selection")
    demo_mode = st.sidebar.radio(
        "Choose a demo:",
        ["🗣️ Vowel Plotting", "❓ Is This a Question?", "📊 Live Spectrogram & Waveform"]
    )

    st.sidebar.markdown("---")
    st.sidebar.markdown("""
    **Tips:**
    - Speak clearly into your microphone
    - Find a quiet environment
    - Try different sounds and see what happens!
    """)

    # =============================================================================
    # DEMO 1: VOWEL PLOTTING (MULTI-VOWEL)
    # =============================================================================
    if demo_mode == "🗣️ Vowel Plotting":
        st.header("🗣️ Vowel Plotting")
        st.markdown("""
        Record multiple vowel sounds and plot them all on a vowel chart to compare your vowel space!
        Each recording will be saved as a separate audio file.
        """)

        # Two-column layout
        col1, col2 = st.columns([1, 2])

        with col1:
            st.subheader("📝 Recording Controls")

            # IPA vowel symbols for copy-paste
            with st.expander("📋 IPA Symbols (click to copy)"):
                st.markdown("""
                **Front vowels:**
                `i` (heed), `ɪ` (hid), `e` (hayed), `ɛ` (head), `æ` (had)

                **Central vowels:**
                `ə` (about), `ʌ` (hut), `ɜ` (bird)

                **Back vowels:**
                `u` (who'd), `ʊ` (hood), `o` (hoed), `ɔ` (hawed), `ɑ` (hod)

                **Tip:** Copy the symbol above and paste into the label field!
                """)

            duration = st.slider("Recording duration (seconds)", 1, 5, 2, key='vowel_duration')
            vowel_label = st.text_input(
                "Vowel label (IPA symbol or word)",
                value="",
                placeholder="e.g., i, æ, or heed",
                key='vowel_label'
            )

            # Record button
            if st.button("🎤 Record Vowel", type="primary", key='record_vowel'):
                if not vowel_label:
                    st.warning("⚠️ Please enter a label for your vowel!")
                else:
                    # Record audio
                    audio, sr = record_audio(duration=duration)
                    st.success("✅ Recording complete!")

                    # Play back audio
                    st.audio(audio, sample_rate=sr)

                    # Extract formants
                    with st.spinner("🔍 Analyzing formants..."):
                        formants = extract_formants(audio, sr)

                    # Save audio file
                    with st.spinner("💾 Saving audio file..."):
                        filename = save_audio_file(audio, sr, vowel_label)

                    # Display formant values
                    if 'F1' in formants and 'F2' in formants:
                        st.success(f"📁 Saved as: `{filename}`")
                        st.metric("F1 (First Formant)", f"{formants['F1']:.0f} Hz")
                        st.metric("F2 (Second Formant)", f"{formants['F2']:.0f} Hz")
                        if 'F3' in formants:
                            st.metric("F3 (Third Formant)", f"{formants['F3']:.0f} Hz")

                        # Add to recordings list
                        st.session_state.vowel_recordings.append({
                            'label': vowel_label,
                            'f1': formants['F1'],
                            'f2': formants['F2'],
                            'f3': formants.get('F3', None),
                            'audio': audio,
                            'sample_rate': sr,
                            'filename': filename
                        })

                        st.success(f"✨ Added '{vowel_label}' to your vowel chart!")
                    else:
                        st.warning("⚠️ Could not extract formants. Try speaking louder or holding the vowel longer.")

            # Display recorded vowels list
            st.markdown("---")
            st.subheader(f"📊 Recorded Vowels ({len(st.session_state.vowel_recordings)})")

            if st.session_state.vowel_recordings:
                for idx, vowel in enumerate(st.session_state.vowel_recordings):
                    with st.container():
                        col_a, col_b = st.columns([3, 1])
                        with col_a:
                            st.write(f"**{idx+1}. {vowel['label']}** - F1: {vowel['f1']:.0f} Hz, F2: {vowel['f2']:.0f} Hz")
                        with col_b:
                            if st.button("🗑️", key=f"delete_{idx}"):
                                st.session_state.vowel_recordings.pop(idx)
                                st.rerun()

                # Clear all button
                if st.button("🗑️ Clear All", key='clear_all'):
                    st.session_state.vowel_recordings = []
                    st.rerun()
            else:
                st.info("No vowels recorded yet. Record some vowels to get started!")

        with col2:
            st.subheader("📈 Vowel Space Chart")

            # Plot vowel chart
            if st.session_state.vowel_recordings:
                fig = plot_vowel_chart_multi(st.session_state.vowel_recordings)
                st.pyplot(fig)

                # Download button for the chart
                import io
                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=300, bbox_inches='tight')
                buf.seek(0)
                st.download_button(
                    label="📥 Download Chart",
                    data=buf,
                    file_name="vowel_chart.png",
                    mime="image/png"
                )
            else:
                # Show empty chart with reference vowels
                fig = plot_vowel_chart_multi(None)
                st.pyplot(fig)
                st.info("👆 Record vowels to see them plotted on the chart!")

    # =============================================================================
    # DEMO 2: PITCH CONTOUR / INTONATION
    # =============================================================================
    elif demo_mode == "❓ Is This a Question?":
        st.header("❓ Is This a Question?")
        st.markdown("""
        Say the same sentence two ways - as a **statement** (falling pitch) and as a
        **question** (rising pitch). See how the pitch contour changes!

        **Try saying:**
        - "You're going to the store" (statement) ⬇️
        - "You're going to the store?" (question) ⬆️
        """)

        duration = st.slider("Recording duration (seconds)", 2, 6, 3)

        col1, col2 = st.columns(2)

        with col1:
            if st.button("🎤 Record Utterance", type="primary"):
                # Record audio
                audio, sr = record_audio(duration=duration)
                st.success("✅ Recording complete!")

                # Play back audio
                st.audio(audio, sample_rate=sr)

                # Extract pitch
                with st.spinner("🔍 Analyzing pitch contour..."):
                    time_points, pitch_values = extract_pitch(audio, sr)

                if len(pitch_values) > 0:
                    # Store in session state
                    st.session_state['pitch_time'] = time_points
                    st.session_state['pitch_values'] = pitch_values

                    # Display pitch statistics
                    st.subheader("Pitch Statistics")
                    st.metric("Mean Pitch", f"{np.mean(pitch_values):.1f} Hz")
                    st.metric("Pitch Range", f"{np.max(pitch_values) - np.min(pitch_values):.1f} Hz")
                else:
                    st.warning("⚠️ Could not extract pitch. Try speaking louder.")

        with col2:
            # Show instructions or stats
            st.info("""
            **What to look for:**
            - 📈 **Rising** pitch at the end = Question
            - 📉 **Falling** pitch at the end = Statement
            - ➡️ **Level** pitch = Flat/monotone
            """)

        # Plot pitch contour
        if 'pitch_time' in st.session_state and 'pitch_values' in st.session_state:
            st.subheader("Pitch Contour")
            fig = plot_pitch_contour(
                st.session_state['pitch_time'],
                st.session_state['pitch_values']
            )
            st.pyplot(fig)
        else:
            st.info("👆 Record your voice to see the pitch contour!")

    # =============================================================================
    # DEMO 3: LIVE SPECTROGRAM & WAVEFORM
    # =============================================================================
    elif demo_mode == "📊 Live Spectrogram & Waveform":
        st.header("📊 Live Spectrogram & Waveform")
        st.markdown("""
        See your voice as a **waveform** (amplitude over time) and **spectrogram**
        (frequency content over time). Try different sounds:

        - **Vowels** - show clear harmonics (horizontal lines)
        - **[s]** sound - high-frequency noise
        - **[ʃ]** "sh" sound - lower frequency than [s]
        - **Singing** - steady pitch shows as horizontal lines
        - **Whispering** - no voicing, just noise
        """)

        duration = st.slider("Recording duration (seconds)", 2, 10, 4)

        if st.button("🎤 Record & Visualize", type="primary"):
            # Record audio
            audio, sr = record_audio(duration=duration)
            st.success("✅ Recording complete!")

            # Play back audio
            st.audio(audio, sample_rate=sr)

            # Plot waveform and spectrogram
            with st.spinner("🎨 Generating visualizations..."):
                fig = plot_waveform_and_spectrogram(audio, sr)

            st.pyplot(fig)

            # Additional info
            st.info("""
            **Reading the visualizations:**
            - **Waveform (top)**: Shows how loud the sound is over time
            - **Spectrogram (bottom)**: Brighter colors = more energy at that frequency
            - Voiced sounds show regular patterns (harmonics)
            - Voiceless sounds [s, sh, f] show noise patterns
            """)


if __name__ == "__main__":
    main()

'''

# Write the app to a file
with open('phonetics_app.py', 'w') as f:
    f.write(app_code)

print("✅ Streamlit app code written to 'phonetics_app.py'")
print("\nTo run the app:")
print("  streamlit run phonetics_app.py")

✅ Streamlit app code written to 'phonetics_app.py'

To run the app:
  streamlit run phonetics_app.py


---
## Testing Individual Components

You can test individual functions here before running the full Streamlit app.

In [10]:
# Optional: Test formant and F0 extraction with a sample audio file
# Uncomment and modify if you have a test audio file

import parselmouth
import numpy as np
from parselmouth.praat import call

def test_formant_extraction(input_wav):
    """Test formant and F0 extraction on a sample audio file."""
    # Load a test audio file
    sound = parselmouth.Sound(input_wav)
    
    # Extract formants
    formant = call(sound, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
    mid_point = sound.duration / 2.0
    
    f1 = call(formant, "Get value at time", 1, mid_point, "Hertz", "Linear")
    f2 = call(formant, "Get value at time", 2, mid_point, "Hertz", "Linear")
    
    print(f"=== FORMANT ANALYSIS FOR {input_wav} ===")
    print(f"F1: {f1:.0f} Hz")
    print(f"F2: {f2:.0f} Hz")
    
    # Extract F0 (pitch)
    pitch = call(sound, "To Pitch", 0.0, 75, 600)  # 75-600 Hz range for human speech
    
    # Get F0 statistics
    mean_f0 = call(pitch, "Get mean", 0.0, 0.0, "Hertz")  # 0.0, 0.0 = entire duration
    min_f0 = call(pitch, "Get minimum", 0.0, 0.0, "Hertz", "Parabolic")
    max_f0 = call(pitch, "Get maximum", 0.0, 0.0, "Hertz", "Parabolic")
    median_f0 = call(pitch, "Get quantile", 0.0, 0.0, 0.50, "Hertz")
    
    print(f"\n=== F0 (PITCH) ANALYSIS FOR {input_wav} ===")
    print(f"Mean F0:   {mean_f0:.1f} Hz")
    print(f"Median F0: {median_f0:.1f} Hz")
    print(f"Min F0:    {min_f0:.1f} Hz")
    print(f"Max F0:    {max_f0:.1f} Hz")
    print(f"F0 Range:  {max_f0 - min_f0:.1f} Hz")
    
    # Get F0 contour over time (optional - shows voicing percentage)
    f0_values = []
    for t in np.arange(0, sound.duration, 0.01):  # Sample every 10ms
        f0 = call(pitch, "Get value at time", t, "Hertz", "Linear")
        if not np.isnan(f0) and f0 > 0:
            f0_values.append(f0)
    
    total_frames = int(sound.duration * 100)
    voiced_percent = (len(f0_values) / total_frames) * 100
    print(f"\nVoiced frames: {len(f0_values)} / {total_frames} ({voiced_percent:.1f}%)")

# Example usage:
test_formant_extraction('./vowel_a.wav')

=== FORMANT ANALYSIS FOR ./vowel_a.wav ===
F1: 725 Hz
F2: 1211 Hz

=== F0 (PITCH) ANALYSIS FOR ./vowel_a.wav ===
Mean F0:   139.3 Hz
Median F0: 134.2 Hz
Min F0:    119.4 Hz
Max F0:    334.4 Hz
F0 Range:  215.0 Hz

Voiced frames: 126 / 279 (45.2%)


---
## Running the App

After executing the code generation cell above, run the Streamlit app from your terminal:

```bash
streamlit run phonetics_app.py
```

The app will open in your browser. You can then:
1. Select a demo from the sidebar
2. Click the record button
3. Speak into your microphone
4. View the analysis and visualizations

### Troubleshooting

**Microphone not working?**
- Check system permissions for microphone access
- Try running: `python -m sounddevice` to test your audio devices

**Poor formant detection?**
- Speak louder and closer to the microphone
- Use a quieter environment
- Hold the vowel sound steady for at least 1 second

**App won't start?**
- Make sure you're in the `phonetics` conda environment
- Verify all packages are installed (run the requirements check cell above)

---

## For Maryland Day Presenters

**Setup checklist:**
- [ ] Test microphone quality in the demo space
- [ ] Have backup audio samples in case mic fails
- [ ] Prepare example sentences for the intonation demo
- [ ] Test on the actual demo computer/laptop beforehand
- [ ] Consider having a touchscreen or wireless mic for audience participation

**Engagement tips:**
- Start with the waveform/spectrogram to show "see your voice"
- Compare different people's vowel spaces (dialect differences!)
- Show McGurk effect videos alongside (YouTube has good ones)
- Have a printed vowel chart for reference

Have fun! 🎉